# เงินลงทุนกับเป้าหมายที่ต้องจ่าย

<p class="lead">มีเงินลงทุน 500,000 บาท จะพอจ่าย 120,000 บาทในอีกสองปี 180,000 บาทในอีกสี่ปี และ 300,000 บาทในอีกหกปีหรือไม่?</p>

การตอบต้องดูทั้งเงินที่มี วันจ่าย และวิธีลงทุนระหว่างรอ เราจะเริ่มจากตารางเงินสามรายการ แล้วคำนวณว่าต้องเตรียมเงินวันนี้เท่าไร ก่อนแยกพอร์ตสำหรับรองรับเป้าหมายออกจากพอร์ตที่ต้องรับความเสี่ยงเพื่อหาโอกาสเติบโต

บทนี้ต่อจากการวัดผลตอบแทนและการจัดพอร์ต โดยอ้างอิงหัวข้อ Module 4 ของคอร์ส EDHEC/Coursera และ Lab 124 ตัวเลขทั้งหมดเป็นตัวอย่างสมมติ หน่วยเป็นบาท ไม่มีภาษี ค่าธรรมเนียม หรือเงินฝากถอนนอกเหนือจากที่ระบุ โค้ดรันได้ใน Notebook ใหม่โดยเรียงจากบนลงล่าง ใช้เพียง NumPy และ pandas ที่ติดตั้งแล้ว

[ดาวน์โหลด Notebook ของบทนี้](https://nutdnuy.github.io/portfolio-management-toolkit/notebooks/asset-liability.ipynb) เพื่อรันตัวอย่างตามลำดับและเทียบผลลัพธ์ที่บันทึกไว้

<span id="goals-as-cashflows"></span>

## เขียนเป้าหมายให้เป็นกระแสเงินสด

Asset คือสินทรัพย์ที่เรามี เช่น เงินฝาก หุ้น หรือพันธบัตร ส่วน liability ในงานบริหารสินทรัพย์และหนี้สินหมายถึงเงินที่ต้องจ่ายในอนาคต อาจเป็นภาระตามสัญญาของกองทุนบำนาญ หรือเป็นเป้าหมายของบุคคล เช่น ค่าเรียนและค่าใช้จ่ายหลังเกษียณ การเรียกเป้าหมายว่า liability ช่วยให้เขียนจำนวนเงินและวันจ่ายลงในแบบจำลองได้ แม้ไม่ได้เป็นหนี้ตามความหมายทางกฎหมาย

เริ่มจากสามรายการดังนี้

| เวลาเมื่อวัดจากวันนี้ | เงินที่ต้องจ่าย | สิ่งที่กำหนดไว้ในตัวอย่าง |
|---:|---:|---|
| อีก 2 ปี | 120,000 บาท | จำนวนเงิน nominal คงที่ |
| อีก 4 ปี | 180,000 บาท | จำนวนเงิน nominal คงที่ |
| อีก 6 ปี | 300,000 บาท | จำนวนเงิน nominal คงที่ |
| รวม | 600,000 บาท | จ่ายคนละวัน |

Nominal หมายถึงจำนวนบาทในวันที่จ่าย เช่น 120,000 บาทตามที่ตกลงไว้ เราจะพิจารณาเป้าหมายที่ปรับตามเงินเฟ้อภายหลัง ยอดรวม 600,000 บาทบอกจำนวนเงินที่ต้องจ่ายตลอดแผน แต่การเตรียมเงินวันนี้อาจใช้น้อยกว่านั้นถ้าสามารถลงทุนรอรับผลตอบแทนตามที่สมมติไว้ได้

วันจ่ายมีผลต่อการเลือกสินทรัพย์ด้วย เงินที่ต้องใช้ปีที่สองไม่ควรถูกนับว่าเป็นเงินที่ถือหุ้นต่อไปได้ถึงปีที่หกโดยไม่มีเงื่อนไข หากขายหุ้นในปีที่สองแล้วขาดทุน ค่าเฉลี่ยผลตอบแทนของทั้งแผนก็ไม่ได้สร้างเงินสดสำหรับวันนั้นเพิ่มขึ้น

In [1]:
import numpy as np
import pandas as pd

liabilities = pd.Series(
    [120_000.0, 180_000.0, 300_000.0],
    index=[2.0, 4.0, 6.0],
    name="Payment in baht",
)
liabilities.index.name = "Years from today"
print(liabilities)
print(f"Total future payments: {liabilities.sum():,.2f}")


Years from today
2.0    120000.0
4.0    180000.0
6.0    300000.0
Name: Payment in baht, dtype: float64
Total future payments: 600,000.00


ตารางมีเงินสามรายการและผลรวม `600,000.00` เครื่องหมาย `_` ภายใน `120_000` มีไว้ช่วยอ่านตัวเลข Python ยังคงมองเป็น 120000 ส่วน `index` บอกว่าจ่ายเมื่อครบกี่ปี ตัวเลข 2.0 เป็นระยะเวลา ไม่ใช่ปีปฏิทิน 2002

ถ้าใช้วันที่จริงต้องตกลงวิธีแปลงวันเป็นเศษส่วนของปีด้วย เช่น จำนวนวันจริงหาร 365 การวัดเวลา 2 ปีพอดีในบทนี้ช่วยให้เห็นหลักการโดยยังไม่ต้องจัดการวันหยุดและวิธีนับวันของตราสารแต่ละชนิด

<span id="present-value"></span>

## คิดย้อนจากเงินวันจ่ายมาหาเงินวันนี้

สมมติว่าสามารถลงทุนได้อัตราทบต้นต่อปี 3% เงิน 100 บาทวันนี้จะเป็น 103 บาทเมื่อครบหนึ่งปี ถ้าต้องการ 103 บาทในหนึ่งปีจึงต้องเตรียม 100 บาทวันนี้ การคิดย้อนนี้เรียกว่า discounting หรือการคิดลด

ให้ $L$ เป็นจำนวนเงินที่จะจ่ายในอีก $T$ ปี และ $y$ เป็นอัตราผลตอบแทนต่อปีแบบ effective ซึ่งนับผลทบต้นครบหนึ่งปีแล้ว มูลค่าปัจจุบันคือ

$$
PV=\frac{L}{(1+y)^T}.
$$

สำหรับเงิน 120,000 บาทในอีกสองปี ที่อัตรา 3% เราต้องเตรียม $120{,}000/1.03^2\approx113{,}111.51$ บาท คูณกลับด้วย $1.03^2$ จะได้เงินวันจ่าย 120,000 บาท

เราใช้ $D(0,T)=(1+y)^{-T}$ แทนตัวคูณคิดลด ตัวนี้ไม่มีหน่วย และบอกมูลค่าวันนี้ของเงิน 1 บาทที่จะได้รับในปีที่ $T$ หากนำไปคูณเงินที่ต้องจ่าย หน่วยของคำตอบจะกลับมาเป็นบาท

In [2]:
annual_yield = 0.03
years = liabilities.index.to_numpy(dtype=float)
discount_factors = (1 + annual_yield) ** (-years)
present_values = liabilities.to_numpy() * discount_factors

pv_table = pd.DataFrame({
    "Payment": liabilities.to_numpy(),
    "Discount factor": discount_factors,
    "Present value": present_values,
}, index=liabilities.index)
liability_pv = present_values.sum()
print(pv_table.round(4))
print(f"Liability PV: {liability_pv:,.2f}")


                   Payment  Discount factor  Present value
Years from today                                          
2.0               120000.0           0.9426    113111.5091
4.0               180000.0           0.8885    159927.6686
6.0               300000.0           0.8375    251245.2770
Liability PV: 524,284.45


ได้มูลค่าปัจจุบันรวม 524,284.45 บาท `to_numpy()` ดึงค่าจาก Series ออกมาเป็น array เพื่อคำนวณจับคู่ตามตำแหน่ง เราสร้างทั้งเวลาและจำนวนเงินจากตารางเดียวกันจึงรักษาลำดับให้ตรงกัน `** (-years)` ยกกำลังติดลบซึ่งเท่ากับการหารด้วยตัวคูณการเติบโต

เงินแต่ละรายการมีเวลารอไม่เท่ากัน จึงต้องคิดลดแยกแล้วบวก การใช้วันจ่ายเฉลี่ยกับยอดรวม 600,000 บาทจะไม่ให้ PV เดียวกันโดยทั่วไป

$$
L_0=\sum_{i=1}^{n}L_iD(0,T_i).
$$

สัญลักษณ์ $L_0$ ในบทนี้หมายถึงมูลค่าปัจจุบันของภาระทั้งหมด และ $L_i$ คือเงินที่จ่ายในวันที่ $T_i$ ถ้าใช้คำว่า liabilities โดยไม่ระบุว่าเป็นยอดวันจ่ายหรือมูลค่าวันนี้ ผู้อ่านอาจนำตัวเลขคนละเวลาไปเทียบกันได้

<span id="discount-curve"></span>

## อัตราคิดลดต้องสอดคล้องกับวันจ่ายและประเภทเงิน

การใช้ 3% กับทุกอายุเรียกว่า flat yield curve หรือเส้นอัตราผลตอบแทนแบน ในตลาดจริงอัตราของเงินสองปี สี่ปี และหกปีอาจต่างกัน เราสามารถใส่อัตรา zero yield ของแต่ละอายุแล้วคิดลดเงินในวันนั้นโดยตรง

Zero yield หมายถึงอัตราที่สอดคล้องกับเงินก้อนเดียวในวันครบกำหนด จึงเหมาะกับการคิดลด cash flow แต่ละรายการ ส่วน yield to maturity ของพันธบัตรมีคูปองสรุปกระแสเงินหลายวันด้วยอัตราเดียว การเอา yield ของพันธบัตรมีคูปองคนละรุ่นมาใส่แทน zero yield โดยตรงอาจไม่ให้ชุด discount factors ที่สอดคล้องกัน

In [3]:
zero_yields = np.array([0.025, 0.030, 0.035])
curve_pv = np.sum(
    liabilities.to_numpy() / (1 + zero_yields) ** years
)
print(f"PV with maturity-specific zero yields: {curve_pv:,.2f}")


PV with maturity-specific zero yields: 518,195.59


ได้ PV รวม 518,195.59 บาท โค้ดใช้อัตรา 2.5% กับเงินปีที่สอง 3% กับเงินปีที่สี่ และ 3.5% กับเงินปีที่หก ผลรวมจะแตกต่างจากการใช้ 3% ทุกอายุ แม้ค่าเฉลี่ยของสามอัตราจะเท่ากับ 3% เพราะทั้งจำนวนเงินและกำหนดจ่ายต่างกัน

อัตราคิดลดสะท้อนข้อตกลงในการประเมินภาระ บทนี้สมมติว่ากระแสเงินแน่นอนเป็นบาทและใช้เส้นอัตราที่สอดคล้องกับการซื้อสินทรัพย์ไม่มีความเสี่ยงผิดนัดในแบบจำลอง หากประเมินตามมาตรฐานบัญชีหรือข้อกำหนดของกองทุน อัตราที่ต้องใช้มีหลักเกณฑ์เฉพาะ จึงต้องตรวจข้อกำหนดนั้นแยกจากสูตรคณิตศาสตร์

การเพิ่มอัตราคิดลดในตารางทำให้ PV ต่ำลง แต่จำนวนเงินที่ต้องจ่าย 600,000 บาทยังเท่าเดิม เช่นเดียวกับการใส่ผลตอบแทนหุ้นคาดหวังสูงขึ้นในแผน การเปลี่ยนสมมติฐานไม่ได้เพิ่มเงินที่มีจริงในบัญชี

<span id="funding-ratio"></span>

## Funding ratio วัดเงินที่มีเทียบกับภาระ

ให้ $A_0$ เป็นมูลค่าสินทรัพย์วันนี้ เราคำนวณ funding ratio และส่วนเกินได้ดังนี้

$$
FR_0=\frac{A_0}{L_0},\qquad S_0=A_0-L_0.
$$

Funding ratio ไม่มีหน่วย เพราะหารบาทด้วยบาท ส่วน surplus หรือส่วนเกินมีหน่วยเป็นบาท หาก $S_0$ ติดลบ เราเรียกขนาดที่ขาดว่า deficit

In [4]:
assets_today = 500_000.0
funding_ratio = assets_today / liability_pv
surplus = assets_today - liability_pv
top_up_today = max(liability_pv - assets_today, 0.0)

print(f"Funding ratio: {funding_ratio:.2%}")
print(f"Surplus: {surplus:,.2f}")
print(f"Top-up for full funding now: {top_up_today:,.2f}")


Funding ratio: 95.37%
Surplus: -24,284.45
Top-up for full funding now: 24,284.45


ได้ funding ratio 95.37% ส่วนเกิน −24,284.45 บาท และเงินเติมเพื่อให้สินทรัพย์เท่ากับ PV ของภาระวันนี้ 24,284.45 บาท คำว่า fully funded ในกรณีนี้หมายถึงสินทรัพย์เพียงพอซื้อกระแสเงินตามราคาและสมมติฐานที่ใช้ ไม่ใช่ความน่าจะเป็นสำเร็จ 100% ของพอร์ตใดก็ได้ที่นำเงินไปลงทุน

ถ้าเงิน 524,284.45 บาทถูกนำไปซื้อสินทรัพย์ที่มี payoff ตรงกับเป้าหมายภายใต้เงื่อนไขที่กำหนด เราติดตามการ matching ได้ แต่ถ้านำเงินจำนวนเดียวกันไปลงทุนหุ้นทั้งหมด เงินอาจลดลงก่อนถึงวันจ่าย แม้ funding ratio ณ วันเริ่มจะเท่ากับ 100%

### สินทรัพย์ไม่ลด แต่ funding ratio ลดได้

ตรึงสินทรัพย์ไว้ที่ 500,000 บาท และประเมินภาระใหม่ที่เวลาเดิม โดยเปลี่ยนอัตราคิดลดเพียงอย่างเดียว

In [5]:
rate_cases = [0.02, 0.03, 0.04]
funding_rows = []
for y in rate_cases:
    value = np.sum(liabilities.to_numpy() / (1 + y) ** years)
    funding_rows.append({
        "Yield": y,
        "Assets": assets_today,
        "Liability PV": value,
        "Funding ratio": assets_today / value,
    })
funding_table = pd.DataFrame(funding_rows)
print(funding_table.round(4))


   Yield    Assets  Liability PV  Funding ratio
0   0.02  500000.0   548023.8451         0.9124
1   0.03  500000.0   524284.4547         0.9537
2   0.04  500000.0   501905.8577         0.9962


| อัตราคิดลด | PV ของภาระ | Funding ratio ของเงิน 500,000 บาท |
|---:|---:|---:|
| 2% | 548,023.85 | 91.24% |
| 3% | 524,284.45 | 95.37% |
| 4% | 501,905.86 | 99.62% |

เมื่ออัตราลดจาก 3% เป็น 2% ต้องใช้เงินวันนี้มากขึ้นเพื่อเตรียมเงินปลายทางเท่าเดิม ดังนั้น funding ratio ลดลงแม้สินทรัพย์คงที่ นี่เป็นการเปรียบเทียบสถานการณ์ ณ เวลาเดียวกัน ยังไม่ใช่เส้นทางที่ผ่านไปปีละหนึ่งขั้น และยังไม่ได้คิดดอกเบี้ยที่บัญชีเงินสดได้รับระหว่างทาง

การมองสินทรัพย์อย่างเดียวจึงอาจทำให้ประเมินความสามารถชำระเป้าหมายคลาดเคลื่อน หากทั้งสินทรัพย์และภาระลดลง ต้องคำนวณอัตราส่วนใหม่ก่อน เช่น สินทรัพย์ลดจาก 100 เป็น 95 แต่ภาระลดจาก 100 เป็น 90 funding ratio จะเพิ่มจาก 100% เป็นประมาณ 105.56%

<span id="nominal-real-goals"></span>

## เป้าหมายเป็นจำนวนบาท หรือเป็นกำลังซื้อ

เงิน 300,000 บาทในอีกหกปีอาจเป็นจำนวนตามสัญญาที่คงที่ แต่ถ้าเราหมายถึง “ค่าครองชีพที่ซื้อสินค้าได้เท่าเงิน 300,000 บาทวันนี้” จำนวนเงินวันจ่ายจะเปลี่ยนตามระดับราคา เป้าหมายแบบหลังเรียกว่า real goal หรือเป้าหมายในหน่วยกำลังซื้อ

สมมติเงินเฟ้อคงที่ 2% ต่อปี เป้าหมายกำลังซื้อ 300,000 บาทวันนี้ต้องใช้เงิน nominal เมื่อครบหกปีเท่ากับ

$$
L_6^{\mathrm{nominal}}=300{,}000(1.02)^6.
$$

ถ้าอัตราทบต้น nominal เท่ากับ 3% อัตรา real ที่สอดคล้องกันภายใต้สมมติฐานเงินเฟ้อคงที่คือ

$$
1+y_{\mathrm{real}}=\frac{1+y_{\mathrm{nominal}}}{1+\pi}.
$$

In [6]:
goal_today_money = 300_000.0
goal_years = 6
inflation = 0.02
nominal_rate = 0.03

goal_future_nominal = goal_today_money * (1 + inflation) ** goal_years
real_rate = (1 + nominal_rate) / (1 + inflation) - 1
pv_nominal_method = goal_future_nominal / (1 + nominal_rate) ** goal_years
pv_real_method = goal_today_money / (1 + real_rate) ** goal_years

print(f"Future nominal goal: {goal_future_nominal:,.2f}")
print(f"Real annual rate: {real_rate:.4%}")
print(f"PV by nominal method: {pv_nominal_method:,.2f}")
print(f"PV by real method: {pv_real_method:,.2f}")
print(np.isclose(pv_nominal_method, pv_real_method))


Future nominal goal: 337,848.73
Real annual rate: 0.9804%
PV by nominal method: 282,942.99
PV by real method: 282,942.99
True


ได้เงินวันจ่ายประมาณ 337,848.73 บาท อัตรา real ประมาณ 0.9804% และสองวิธีให้ PV เท่ากันที่ 282,942.99 บาท คำสั่ง `np.isclose` ตรวจว่าค่าทั้งสองใกล้กันในระดับความคลาดเคลื่อนที่เหมาะกับเลขทศนิยมของคอมพิวเตอร์

เงื่อนไขเงินเฟ้อคงที่ทำให้ตัวอย่างนี้คำนวณง่าย เมื่อเงินเฟ้อไม่แน่นอน เป้าหมาย nominal ก็ไม่แน่นอนด้วย การซื้อพันธบัตร nominal ที่ครบหกปีจะล็อกจำนวนบาทตามสัญญา แต่กำลังซื้อของเงินนั้นยังเปลี่ยนได้

ตราสารที่ปรับตามดัชนีเงินเฟ้ออาจช่วยรับความเสี่ยงด้านนี้ เช่น TIPS ของสหรัฐปรับเงินต้นตาม CPI และคำนวณดอกเบี้ยจากเงินต้นที่ปรับแล้ว อย่างไรก็ตาม TIPS อ้างอิงเงินเฟ้อสหรัฐและจ่ายดอลลาร์ จึงไม่ตรงกับค่าเรียนหรือค่าครองชีพสกุลบาทโดยอัตโนมัติ ความคลาดเคลื่อนระหว่างดัชนี สกุลเงิน และค่าใช้จ่ายจริงเป็น basis risk [รายละเอียด TIPS จาก TreasuryDirect](https://www.treasurydirect.gov/marketable-securities/tips/)

<span id="goal-hedging-portfolio"></span>

## สร้างพอร์ตที่จ่ายเงินตรงกับเป้าหมาย

Zero-coupon bond หรือพันธบัตรไม่มีคูปองจ่ายเงินก้อนเดียวเมื่อครบกำหนด สมมติว่ามีตราสารไม่มีความเสี่ยงผิดนัดทุกอายุที่ต้องการ ซื้อขายได้ตามจำนวนที่ต้องการ และใช้ discount factors ชุดเดียวกับที่ประเมินภาระ เราซื้อเงินต้นครบกำหนด 120,000 บาทที่ปี 2 อีก 180,000 บาทที่ปี 4 และ 300,000 บาทที่ปี 6 ได้ด้วยต้นทุนรวมเท่ากับ $L_0$

นี่คือ cash-flow matching: วันที่และจำนวนเงินรับจากสินทรัพย์ตรงกับเงินที่ต้องจ่าย สำหรับบุคคลเรียกพอร์ตเช่นนี้ว่า goal-hedging portfolio หรือ GHP และในบริบทภาระของสถาบันเรียกว่า liability-hedging portfolio หรือ LHP

หากแต่ละหน่วยจ่ายเงิน 1 บาท ณ วันครบกำหนด ราคาต่อหน่วยคือ discount factor จำนวนหน่วยที่ซื้อเท่ากับเงินเป้าหมาย ไม่ใช่จำนวนบาทที่ใช้ซื้อวันนี้

In [7]:
bond_units = liabilities.to_numpy()
purchase_costs = bond_units * discount_factors
cash_received = bond_units * np.ones(len(bond_units))

matching_table = pd.DataFrame({
    "ZCB units": bond_units,
    "Price per unit": discount_factors,
    "Cost today": purchase_costs,
    "Cash at own maturity": cash_received,
    "Required payment": liabilities.to_numpy(),
}, index=liabilities.index)
print(matching_table.round(4).to_string())
print(np.allclose(cash_received, liabilities.to_numpy()))


                  ZCB units  Price per unit   Cost today  Cash at own maturity  Required payment
Years from today                                                                                
2.0                120000.0          0.9426  113111.5091              120000.0          120000.0
4.0                180000.0          0.8885  159927.6686              180000.0          180000.0
6.0                300000.0          0.8375  251245.2770              300000.0          300000.0
True


ผลตรวจเป็น `True` เราไม่ได้นำพันธบัตรปีที่สองไปถือจนปีที่หก เงินของมันถูกใช้จ่ายเมื่อครบปีที่สองตามแผน การประเมิน funding ratio หลังจ่ายแล้วต้องเอาทั้งเงินจ่ายและ liability ที่ชำระเสร็จออกพร้อมกัน

### ราคาพันธบัตรแกว่ง แต่สัดส่วนการรองรับภาระคงที่ได้

พิจารณาเฉพาะเป้าหมายหนึ่งรายการ จ่าย 100,000 บาทในอีกห้าปี ถ้าราคา ZCB ต่อเงินครบกำหนด 1 บาทวันนี้คือ $P(0,5)$ มูลค่าภาระคือ $100{,}000P(0,5)$ การถือ ZCB 100,000 หน่วยจึงมีสินทรัพย์เท่ากับภาระทุกครั้งที่ใช้ราคาเดียวกันประเมินทั้งสองฝั่ง

ถ้ามีทุนเพียง 90% ของต้นทุน ซื้อได้ 90,000 หน่วย funding ratio จะคงอยู่ที่ 90% ภายใต้การ matching นี้ การ hedge รักษาสัดส่วนที่เริ่มต้นไว้ แต่ไม่ได้เปลี่ยนเงินที่ขาดให้กลายเป็นเงินที่มี

In [8]:
single_goal = 100_000.0
remaining_years = np.array([5.0, 4.0, 3.0])
scenario_yields = np.array([0.03, 0.02, 0.04])
single_bond_prices = (1 + scenario_yields) ** (-remaining_years)
matched_units = 0.90 * single_goal
matched_assets = matched_units * single_bond_prices
single_liability_values = single_goal * single_bond_prices

print(pd.DataFrame({
    "Remaining years": remaining_years,
    "Bond price": single_bond_prices,
    "Assets": matched_assets,
    "Liability PV": single_liability_values,
    "Funding ratio": matched_assets / single_liability_values,
}).round(4))


   Remaining years  Bond price      Assets  Liability PV  Funding ratio
0              5.0      0.8626  77634.7906    86260.8784            0.9
1              4.0      0.9238  83146.0883    92384.5426            0.9
2              3.0      0.8890  80009.6723    88899.6359            0.9


ทั้งสามแถวให้ funding ratio 0.9 หรือ 90% ตัวอย่างกำหนดราคาแต่ละวันจากเส้นอัตราแบนของวันนั้นเพื่อให้เห็นเอกลักษณ์ทางพีชคณิต ไม่ได้จำลองว่าอัตราดอกเบี้ยจะเคลื่อนไหวแบบใดจริง

ในทางปฏิบัติอาจไม่มี ZCB ครบทุกอายุ ตราสารอาจผิดนัด มีต้นทุนซื้อขาย หรือภาระเปลี่ยนจากการประมาณค่าใช้จ่ายใหม่ หากจับคู่ cash flow ไม่ได้ตรงทั้งหมด เราอาจจับคู่ความไวต่ออัตราดอกเบี้ยด้วย duration ซึ่งจะคำนวณในบท [พันธบัตรและ Duration](https://nutdnuy.github.io/portfolio-management-toolkit/bonds-duration.html)

<span id="psp-ghp"></span>

## แยกหน้าที่ของ PSP และ GHP

Performance-seeking portfolio หรือ PSP คือส่วนที่รับความเสี่ยงเพื่อหาโอกาสเติบโต อาจเป็นพอร์ตหุ้นและสินทรัพย์อื่นที่กระจายความเสี่ยงแล้ว ส่วน GHP จัดตามวันจ่าย สกุลเงิน และความเสี่ยงของเป้าหมาย การแบ่งนี้เป็นการแยกหน้าที่ ไม่ได้กำหนดว่าทุกคนต้องถือหุ้นกี่เปอร์เซ็นต์

เราประเมิน PSP ด้วยผลตอบแทน ความเสี่ยง ต้นทุน และความน่าเชื่อถือของสมมติฐาน ส่วน GHP ต้องดูว่าจ่ายเงินตรงหรือเคลื่อนไหวสอดคล้องกับมูลค่าเป้าหมายเพียงใด การจัดอันดับ GHP ด้วย Sharpe ratio เพียงตัวเดียวอาจพาให้เลือกพอร์ตที่ผลตอบแทนดีแต่รับความเสี่ยงคนละชนิดกับภาระ

สมมติมีเงิน 100,000 บาท และเป้าหมายขั้นต่ำ 100,000 บาทในอีกห้าปี อัตราที่ล็อกได้เท่ากับ 3% ต่อปี การซื้อ ZCB รองรับเป้าหมายทั้งจำนวนใช้ประมาณ 86,260.88 บาท เหลือประมาณ 13,739.12 บาทให้จัดสรรส่วนอื่นได้ ภายใต้สมมติฐานว่าเงินส่วนอื่นไม่สร้างภาระขาดทุนเกินจำนวนที่ลงทุน

In [9]:
starting_assets = 100_000.0
minimum_goal = 100_000.0
horizon = 5
locked_yield = 0.03
ghp_cost = minimum_goal / (1 + locked_yield) ** horizon
remaining_budget = starting_assets - ghp_cost
print(f"Cost to match the minimum goal: {ghp_cost:,.2f}")
print(f"Budget left over: {remaining_budget:,.2f}")


Cost to match the minimum goal: 86,260.88
Budget left over: 13,739.12


เงินที่เหลือในตัวอย่างนี้เป็นจำนวนที่คำนวณได้จากเป้าหมายและราคา hedge หากลงทุน PSP ด้วยเงินกู้หรือเครื่องมือที่สร้างภาระเพิ่มเติม การกันเงินใน GHP อาจไม่เพียงพอเมื่อเจ้าหนี้มีสิทธิเรียกร้องสินทรัพย์รวม จึงต้องกำหนดขอบเขตของพอร์ตและสัญญาด้วย

ถ้าต้นทุน GHP สูงกว่าเงินที่มี การเลือก PSP ให้เสี่ยงขึ้นอาจเพิ่มโอกาสบางเส้นทางที่จะถึงเป้าหมาย พร้อมเพิ่มโอกาสขาดทุนในเส้นทางอื่น ผู้ลงทุนยังต้องพิจารณาเพิ่มเงินออม ลดจำนวนเป้าหมาย เลื่อนเวลา หรือยอมรับความเสี่ยงที่กำหนดไว้อย่างชัดเจน

<span id="funding-risk-budget"></span>

## งบเงินและงบความเสี่ยงต้องดูร่วมกัน

การจัดสรร 60% ไป PSP ไม่ได้บอกความเสี่ยงครบถ้วน PSP สองก้อนที่มี volatility ต่างกันย่อมส่งผลต่อแผนไม่เท่ากัน และ GHP สองก้อนที่จับคู่ภาระได้ต่างกันก็ทำให้ funding ratio แกว่งไม่เท่ากัน

Risk budget จึงควรเขียนเป็นผลที่ยอมรับได้ เช่น ความถี่จำลองที่เงินปลายทางต่ำกว่าเป้าหมาย จำนวนเงินที่ขาดเมื่อขาดจริง หรือขนาด drawdown ของ funding ratio ระหว่างทาง ถ้าเป้าหมายมีเงินจ่ายทุกปี ต้องตรวจสภาพคล่องแต่ละปีด้วย การจบปีสุดท้ายด้วยเงินพอไม่ได้พิสูจน์ว่ามีเงินจ่ายครบก่อนหน้านั้น

เงินเฟ้อ อายุขัย และค่าใช้จ่ายสุขภาพทำให้ภาระของบุคคลไม่แน่นอน การกำหนดเพียงค่าใช้จ่ายยี่สิบปีแล้วรายงานว่าแผนครบถ้วนเท่ากับละความเสี่ยงหลังปีที่ยี่สิบไว้ข้างนอกแบบจำลอง ต้องระบุส่วนที่จำลองและเงินรองรับส่วนที่ยังไม่จำลองให้ชัด

ในบท [จัดสรรเงินตามเป้าหมาย](https://nutdnuy.github.io/portfolio-management-toolkit/goal-based-allocation.html) เราจะเปรียบเทียบสัดส่วนคงที่ การลดความเสี่ยงตามเวลา และการปรับตามส่วนเผื่อ โดยรายงานทั้งเงินปลายทางและการขาดเป้าหมายจากทุกเส้นทาง

<span id="asset-liability-exercises"></span>

## แบบฝึกหัดพร้อมวิธีตรวจ

1. ต้องจ่าย 103,000 บาทในอีกหนึ่งปี และล็อกอัตรา 3% ได้ ต้องมีเงินวันนี้เท่าไร?
2. มีสินทรัพย์ 90,000 บาท และ PV ของภาระ 100,000 บาท funding ratio กับ deficit เป็นเท่าไร?
3. หากสินทรัพย์คงที่แต่ดอกเบี้ยลดลง เหตุใด funding ratio ของเป้าหมายเงินคงที่จึงอาจลดลง?
4. เป้าหมายคือค่าครองชีพเทียบเท่า 100,000 บาทวันนี้ในอีกสองปี เงินเฟ้อคงที่ 2% ต้องตั้งเงิน nominal เท่าไร?
5. การถือ ZCB ตรงอายุด้วยเงิน 80% ของ PV เป้าหมายช่วยให้ funding ratio กลายเป็น 100% ได้หรือไม่?
6. ผู้ลงทุนมีหนี้จ่ายบาท แต่ GHP เป็นตราสารดอลลาร์ที่จ่ายวันตรงกัน ยังเหลือความเสี่ยงอะไร?

<details>
<summary>เปิดเฉลยและเหตุผล</summary>

ข้อ 1 ใช้ $103{,}000/1.03=100{,}000$ บาท ข้อ 2 funding ratio 90% และ deficit 10,000 บาท โดย surplus เท่ากับ −10,000 บาท

ข้อ 3 เมื่อคิดลดน้อยลงต้องเตรียมเงินวันนี้มากขึ้น ตัวส่วนของ funding ratio จึงเพิ่ม ส่วนข้อ 4 เงิน nominal เท่ากับ $100{,}000(1.02)^2=104{,}040$ บาท การใช้เป้าหมาย 100,000 บาทคงที่ในวันจ่ายจะได้กำลังซื้อน้อยกว่าที่กำหนด

ข้อ 5 การ matching รักษาสัดส่วน 80% ภายใต้สมมติฐานของแบบจำลอง ต้องเติมเงินหรือเปลี่ยนเป้าหมายจึงจะซื้อ cash flow ได้ครบ ข้อ 6 ยังมีความเสี่ยงอัตราแลกเปลี่ยน รวมถึงความเสี่ยงผิดนัดและความต่างของสัญญา หากไม่ได้กำจัดด้วยวิธีอื่นอย่างสอดคล้องกัน

</details>

<span id="asset-liability-sources"></span>

## แหล่งเรียนและขอบเขต

อ่าน Transcript ของ [From Asset Management to Asset-Liability Management](https://www.coursera.org/learn/introduction-portfolio-construction-python/lecture/x66ru/from-asset-management-to-asset-liability-management), [Liability hedging portfolios](https://www.coursera.org/learn/introduction-portfolio-construction-python/lecture/JT5zQ/liability-hedging-portfolios) และ [Liability-driven investing](https://www.coursera.org/learn/introduction-portfolio-construction-python/lecture/n0bF9/liability-driven-investing-ldi) ประกอบ Notebook `lab_124.ipynb` ที่ผู้เรียนให้มา ตัวเลขและโค้ดในหน้านี้เป็นตัวอย่างสมมติที่เรียบเรียงขึ้นใหม่

การ matching ในตัวอย่างอาศัย cash flow แน่นอน เส้นอัตราที่สอดคล้องกัน และตราสารที่จ่ายได้ตามสัญญา การใช้งานที่มีภาระหลายประเภทต้องกำหนดวิธีประเมินและความเสี่ยงที่ hedge ไม่ได้เพิ่ม ดูกรอบเรื่อง immunization ใน [CFA Institute: Liability-Driven and Index-Based Strategies](https://www.cfainstitute.org/insights/professional-learning/refresher-readings/2026/liability-driven-index-based-strategies)